# Experiment 09: All-26-Layers Branch 1 with Tensor Caching, Gradient Descent Core Finding & 99% Compression Sweep

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25].mlp.gate_proj`)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Key Pipeline Principles:
1. **Single-Pass 26-Layer Profiling**: Simultaneous forward hooks across all 26 decoder layers during uncompressed baseline inference ($W_{\text{orig}}$).
2. **Tensor Assembly & Memory Caching**: Pre-assembles all 26 layer tensors $\mathcal{T}_l \in \mathbb{R}^{6 \times 400 \times 1152}$ into a list `cached_layer_tensors = [T_0, ..., T_25]` for fast multi-rate experimentation without re-profiling.
3. **Layer-Wise Superweight Quarantine**: Extreme outliers ($|x| > 3.0$ or top 1% variance) preserved in pristine uncompressed FP32.
4. **Gradient Descent Core Finding (Adam Optimization)**:
   Initializes Tucker factors via SVD/HOSVD, then optimizes the core tensor $\mathcal{S}$ and factor matrices using PyTorch Adam gradient descent to minimize reconstruction error:
   $$\min_{\mathcal{S}, A, B, C} \|\mathcal{T}_l - \mathcal{S} \times_1 A \times_2 B \times_3 C\|_F^2$$
5. **Compression Sweep Starting from ~99%**:
   Evaluates compression tiers starting from ultra-high parameter cut (98.8% cut, ranks $[2, 20, 20]$), down through 90% and moderate ranks.
6. **Full-Model Evaluation**: Benchmarks all 26 compressed layers simultaneously on GLUE MNLI.

In [1]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

/home/dwithun/Development/llm_compression/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded neural_decomp library.
Device: NVIDIA GeForce RTX 3070 Ti | CUDA Available: True


In [2]:
# =====================================================================
# STEP 2: Initialize Model & Tokenizer Across All 26 Layers
# =====================================================================
model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

NUM_LAYERS = len(model.model.layers)
print(f"Loaded model with {NUM_LAYERS} transformer decoder layers.")

# Save pristine unprocessed weights for all 26 gate_proj layers for rollbacks
W_gate_orig_all = {
    l: model.model.layers[l].mlp.gate_proj.weight.data.clone()
    for l in range(NUM_LAYERS)
}
print(f"Stored pristine baseline weights for all {NUM_LAYERS} gate_proj modules.")

Loading weights: 100%|██████████| 340/340 [00:03<00:00, 92.05it/s] 


Loaded model with 26 transformer decoder layers.
Stored pristine baseline weights for all 26 gate_proj modules.
time: 7.05s
cummulative_time: 9.44s


In [3]:
# =====================================================================
# STEP 3: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

# Calibration & benchmark evaluation subset
EVAL_SAMPLE_COUNT = 1000
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Active Evaluation Subset: {len(eval_data):,} samples")

Loaded GLUE MNLI: 9,815 total samples | Active Evaluation Subset: 1,000 samples
time: 3.65s
cummulative_time: 13.10s


## Step 1: Single-Pass Model-Wide Activation Profiling Across All 26 Layers

We attach forward hooks across all 26 decoder layers during uncompressed baseline inference.
We capture the empirical activation trajectories and compute the pristine baseline accuracy on GLUE MNLI.

In [4]:
# =====================================================================
# STEP 4: Initial Baseline Evaluation & Simultaneous 26-Layer Hooking
# =====================================================================
layer_trajectories = {l: [] for l in range(NUM_LAYERS)}
current_acts = {}

def make_act_hook(layer_idx):
    def hook_fn(module, input_tensor, output_tensor):
        act = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
        current_acts[layer_idx] = act.detach().cpu()
    return hook_fn

hooks = [
    model.model.layers[l].mlp.act_fn.register_forward_hook(make_act_hook(l))
    for l in range(NUM_LAYERS)
]

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline Eval & 26-Layer Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        for l in range(NUM_LAYERS):
            if l in current_acts and current_acts[l] is not None:
                pooled = current_acts[l].squeeze(0).mean(dim=0).numpy()
                layer_trajectories[l].append(pooled)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

for h in hooks:
    h.remove()

acts_matrices = {l: np.stack(layer_trajectories[l]) for l in range(NUM_LAYERS)}

baseline_accuracy = accuracy_score(ground_truth, predictions)
print(f"\nUncompressed Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Profiled activation trajectories across all {NUM_LAYERS} layers.")

Baseline Eval & 26-Layer Profiling: 100%|██████████| 1000/1000 [00:45<00:00, 21.74it/s]



Uncompressed Baseline Accuracy: 48.00%
Profiled activation trajectories across all 26 layers.
time: 46.19s
cummulative_time: 59.29s


## Step 2: Pre-Assemble & Cache All 26 Layer Tensors in Memory

Rather than re-computing coordinate groupings or re-profiling the model, we assemble all 26 layer tensors:
$$\mathcal{T}_l \in \mathbb{R}^{6 \times 400 \times 1152} \quad (l = 0 \dots 25)$$
and store them in `cached_layer_tensors = [T_0, ..., T_25]`.
Each tensor is ~11 MB on CPU (~286 MB total for all 26 layers), allowing instant parameter sweeps and rank exploration.

In [5]:
# =====================================================================
# STEP 5: Assemble & Cache All 26 Layer Tensors into a List
# =====================================================================
NUM_CLUSTERS = 6
COORDS_PER_CLUSTER = 400
TARGET_TOTAL_ACTIVE = NUM_CLUSTERS * COORDS_PER_CLUSTER

cached_layer_tensors = []
layer_metadata = []

for l in range(NUM_LAYERS):
    acts_l = acts_matrices[l]
    W_gate_l = W_gate_orig_all[l]
    
    # 1. Outlier Filter
    max_mags = np.max(np.abs(acts_l), axis=0)
    variances = np.var(acts_l, axis=0)
    super_mask = (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]
    
    inactive_mask = (np.mean(np.abs(acts_l) < 0.05, axis=0) > 0.90) & (~super_mask)
    normal_active_indices = np.where((~super_mask) & (~inactive_mask))[0]
    
    # 2. 1D Top Frequent Values (Branch 1)
    rounded_acts = np.round(acts_l[:, normal_active_indices], decimals=1)
    top_vals = []
    for idx in range(len(normal_active_indices)):
        vals, counts = np.unique(rounded_acts[:, idx], return_counts=True)
        top_vals.append(vals[np.argmax(counts)])
    top_vals = np.array(top_vals)
    
    # 3. Mutually Exclusive 1D Absolute Difference Grouping
    sorted_order = np.argsort(top_vals)[:TARGET_TOTAL_ACTIVE]
    selected_coords = normal_active_indices[sorted_order]
    
    cluster_slices = []
    clusters_dict = {}
    for k in range(NUM_CLUSTERS):
        c_coords = selected_coords[k * COORDS_PER_CLUSTER : (k + 1) * COORDS_PER_CLUSTER]
        clusters_dict[k] = c_coords
        cluster_slices.append(W_gate_l[c_coords, :].float().cpu())
        
    T_l = torch.stack(cluster_slices, dim=0)
    cached_layer_tensors.append(T_l)
    
    layer_metadata.append({
        "layer": l,
        "super_indices": super_indices,
        "clusters_dict": clusters_dict,
        "d_in": W_gate_l.shape[1],
        "num_coords": W_gate_l.shape[0],
    })

print(f"Successfully assembled and cached all {len(cached_layer_tensors)} layer tensors in memory.")
print(f"Tensor shape per layer: {list(cached_layer_tensors[0].shape)} ({cached_layer_tensors[0].numel():,} params)")

Successfully assembled and cached all 26 layer tensors in memory.
Tensor shape per layer: [6, 400, 1152] (2,764,800 params)
time: 3.14s
cummulative_time: 62.43s


## Step 3: Gradient Descent Core Finding & Factor Optimization

We define `optimize_tucker_gd()`:
1. Initializes factor matrices $A, B, C$ and core $\mathcal{S}$ using Higher-Order SVD (`init='svd'`).
2. Converts the core $\mathcal{S}$ and factor matrices into PyTorch trainable parameters.
3. Applies **Adam gradient descent** (learning rate `1e-3`, 30 steps) minimizing Frobenius reconstruction loss:
   $$\mathcal{L} = \|\mathcal{T} - \mathcal{S} \times_1 A \times_2 B \times_3 C\|_F^2$$
This refines the core tensor to minimize approximation error and structural noise.

In [6]:
# =====================================================================
# STEP 6: Define Gradient Descent Core Finding Function
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=30, lr=1e-3, device="cpu"):
    """
    Performs Tucker decomposition with SVD initialization followed by
    gradient descent refinement of the core tensor and factor matrices.
    """
    # 1. HOSVD Initialization
    core_init, factors_init = tucker(T, rank=ranks, init='svd')
    
    # 2. Convert to trainable PyTorch parameters
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)
    
    # Initial error
    with torch.no_grad():
        T_recon_init = tucker_to_tensor((core_param, factors_param))
        init_err = (torch.norm(T_target - T_recon_init) / torch.norm(T_target)).item()
        
    # 3. Gradient Descent Optimization
    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()
        
    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param)).cpu()
        final_err = (torch.norm(T.cpu() - T_recon_final) / torch.norm(T.cpu())).item()
        
    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, init_err, final_err

# Quick demo on Layer 0 cached tensor
demo_ranks = [4, 180, 600]
core_opt, factors_opt, T_demo_recon, svd_err, gd_err = optimize_tucker_gd(cached_layer_tensors[0], demo_ranks, num_steps=30)
print(f"Layer 0 GD Core Optimization Demo (Ranks {demo_ranks}):")
print(f"  SVD Init Recon Error: {svd_err * 100:.2f}%")
print(f"  GD Refined Recon Error: {gd_err * 100:.2f}% (Loss minimized via Adam)")

Layer 0 GD Core Optimization Demo (Ranks [4, 180, 600]):
  SVD Init Recon Error: 77.17%
  GD Refined Recon Error: 77.16% (Loss minimized via Adam)
time: 2.07s
cummulative_time: 64.51s


## Step 4: Multi-Tier Compression Sweep Starting from ~99%

We define compression tiers starting from ultra-high parameter cut down to balanced ranks:
- **Tier 1: Ultra-High (~99% cut / 86x on active tensor)**: Ranks $[2, 20, 20]$
- **Tier 2: Aggressive (~89% cut / 8.9x on active tensor)**: Ranks $[3, 80, 200]$
- **Tier 2.5: Target (~77% cut / 4.35x on active tensor)**: Ranks $[4, 120, 360]$
- **Tier 3: Moderate (~57% cut / 2.3x on active tensor)**: Ranks $[4, 180, 600]$

We apply Gradient Descent core finding to all 26 cached tensors and evaluate the compression tiers.

In [7]:
# =====================================================================
# STEP 7: Run GD-Tucker Decomposition Across Compression Tiers
# =====================================================================
compression_tiers = [
    {"label": "Tier 1 (Ultra ~99% Cut)", "ranks": [2, 20, 20]},
    {"label": "Tier 2 (Aggressive ~89% Cut)", "ranks": [3, 80, 200]},
    {"label": "Tier 2.5 (Target ~77% Cut)",   "ranks": [4, 120, 360]},
    {"label": "Tier 3 (Moderate ~57% Cut)",   "ranks": [4, 180, 600]},
]

# We will test each tier on the full model
tier_results = []

for tier_cfg in compression_tiers:
    tier_label = tier_cfg["label"]
    ranks = tier_cfg["ranks"]
    print(f"\n{'='*75}")
    print(f"Executing: {tier_label} with Ranks {ranks} across all 26 layers...")
    print(f"{'='*75}")
    
    total_orig_gate = 0
    total_comp_gate = 0
    layer_errors = []
    
    # Process all 26 layers using cached tensors and GD core finding
    for l in range(NUM_LAYERS):
        T_l = cached_layer_tensors[l]
        meta = layer_metadata[l]
        
        # Optimize core & factors via Adam GD
        core_opt, factors_opt, T_l_recon, svd_err, gd_err = optimize_tucker_gd(
            T_l, ranks, num_steps=25, lr=1e-3
        )
        layer_errors.append(gd_err)
        
        # Parameter accounting
        core_params = core_opt.numel()
        factor_params = sum(f.numel() for f in factors_opt)
        comp_active = core_params + factor_params
        orig_active = T_l.numel()
        
        full_orig = W_gate_orig_all[l].numel()
        full_comp = full_orig - orig_active + comp_active
        total_orig_gate += full_orig
        total_comp_gate += full_comp
        
        # Inject reconstructed weights into live model Layer l
        W_reconstructed = W_gate_orig_all[l].clone().float().cpu()
        for k in range(NUM_CLUSTERS):
            W_reconstructed[meta["clusters_dict"][k], :] = T_l_recon[k].float().cpu()
        W_reconstructed[meta["super_indices"], :] = W_gate_orig_all[l][meta["super_indices"], :].float().cpu()
        
        target_mod = model.model.layers[l].mlp.gate_proj
        target_mod.weight.data = W_reconstructed.to(device=model.device, dtype=target_mod.weight.dtype)
        
    global_cut_pct = (1.0 - total_comp_gate / total_orig_gate) * 100.0
    global_ratio = total_orig_gate / total_comp_gate
    mean_err = np.mean(layer_errors) * 100.0
    
    print(f"Global Gate Params: {total_comp_gate:,} vs {total_orig_gate:,} ({global_ratio:.2f}x, {global_cut_pct:.2f}% cut)")
    print(f"Mean Layer Recon Error: {mean_err:.2f}%")
    
    # Evaluate full model on GLUE MNLI
    print(f"Evaluating {tier_label} on GLUE MNLI...")
    tier_preds, tier_gts = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Eval {tier_label}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs)
            next_token_logits = outputs.logits[0, -1, :]
            pred_label = torch.argmax(next_token_logits[label_token_ids]).item()
            tier_preds.append(pred_label)
            tier_gts.append(sample["label"])
            
    tier_acc = accuracy_score(tier_gts, tier_preds)
    delta = tier_acc - baseline_accuracy
    print(f"  Accuracy: {tier_acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    
    tier_results.append({
        "Tier": tier_label,
        "Ranks": str(ranks),
        "Total Params": total_comp_gate,
        "Ratio": f"{global_ratio:.2f}x",
        "Cut %": f"{global_cut_pct:.2f}%",
        "Mean Err %": f"{mean_err:.2f}%",
        "Accuracy %": f"{tier_acc * 100:.2f}%",
        "Delta": f"{delta * 100:+.2f}%",
    })

# Restore clean baseline
for l in range(NUM_LAYERS):
    model.model.layers[l].mlp.gate_proj.weight.data = W_gate_orig_all[l].clone()
print("\nAll 26 layer baseline weights restored.")


Executing: Tier 1 (Ultra ~99% Cut) with Ranks [2, 20, 20] across all 26 layers...
Global Gate Params: 135,971,576 vs 207,028,224 (1.52x, 34.32% cut)
Mean Layer Recon Error: 97.84%
Evaluating Tier 1 (Ultra ~99% Cut) on GLUE MNLI...


Eval Tier 1 (Ultra ~99% Cut): 100%|██████████| 1000/1000 [00:32<00:00, 30.31it/s]


  Accuracy: 34.10% (Δ vs Baseline: -13.90%)

Executing: Tier 2 (Aggressive ~89% Cut) with Ranks [3, 80, 200] across all 26 layers...
Global Gate Params: 143,214,292 vs 207,028,224 (1.45x, 30.82% cut)
Mean Layer Recon Error: 90.55%
Evaluating Tier 2 (Aggressive ~89% Cut) on GLUE MNLI...


Eval Tier 2 (Aggressive ~89% Cut): 100%|██████████| 1000/1000 [00:34<00:00, 29.06it/s]


  Accuracy: 37.10% (Δ vs Baseline: -10.90%)

Executing: Tier 2.5 (Target ~77% Cut) with Ranks [4, 120, 360] across all 26 layers...
Global Gate Params: 151,667,568 vs 207,028,224 (1.37x, 26.74% cut)
Mean Layer Recon Error: 83.55%
Evaluating Tier 2.5 (Target ~77% Cut) on GLUE MNLI...


Eval Tier 2.5 (Target ~77% Cut): 100%|██████████| 1000/1000 [00:35<00:00, 27.85it/s]


  Accuracy: 32.00% (Δ vs Baseline: -16.00%)

Executing: Tier 3 (Moderate ~57% Cut) with Ranks [4, 180, 600] across all 26 layers...
Global Gate Params: 166,219,248 vs 207,028,224 (1.25x, 19.71% cut)
Mean Layer Recon Error: 76.30%
Evaluating Tier 3 (Moderate ~57% Cut) on GLUE MNLI...


Eval Tier 3 (Moderate ~57% Cut): 100%|██████████| 1000/1000 [00:33<00:00, 29.58it/s]

  Accuracy: 32.90% (Δ vs Baseline: -15.10%)

All 26 layer baseline weights restored.
time: 270.13s
cummulative_time: 334.70s


## Step 5: Summary Table, Parameter Reduction & Artifact Export

We tabulate the multi-tier compression sweep results, parameter reduction across all 26 layers, and final benchmark retention.

In [8]:
# =====================================================================
# STEP 8: Summary Table & Artifact Export
# =====================================================================
summary_rows = [
    {
        "Tier": "Baseline (Uncompressed)",
        "Ranks": "Full",
        "Total Params": 26 * 6912 * 1152,
        "Ratio": "1.00x",
        "Cut %": "0.00%",
        "Mean Err %": "0.00%",
        "Accuracy %": f"{baseline_accuracy * 100:.2f}%",
        "Delta": "+0.00%",
    }
] + tier_results

print("=" * 105)
print(f"{'Tier':<30} | {'Ranks':<15} | {'Params':<10} | {'Ratio':<6} | {'Cut %':<7} | {'Err %':<8} | {'Accuracy':<9} | {'Delta':<7}")
print("=" * 105)
for r in summary_rows:
    print(f"{r['Tier']:<30} | {r['Ranks']:<15} | {r['Total Params']:<10} | {r['Ratio']:<6} | {r['Cut %']:<7} | {r['Mean Err %']:<8} | {r['Accuracy %']:<9} | {r['Delta']:<7}")
print("=" * 105)

# Global Notebook Timing
total_notebook_runtime = time.time() - GLOBAL_NOTEBOOK_START_TIME
print(f"cummulative_time: {total_notebook_runtime:.2f}s")

# Save export artifact
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
output_path = artifact_dir / "09_all_layers_gd_branch1_results.json"

export_data = {
    "experiment": "09_all_layers_gd_core_branch1",
    "model_id": model_id,
    "num_layers": NUM_LAYERS,
    "eval_samples": len(eval_data),
    "baseline_accuracy": baseline_accuracy,
    "results": summary_rows,
    "timing_summary": {
        "cummulative_time_sec": round(total_notebook_runtime, 3),
        "cell_timings": NOTEBOOK_TIMINGS,
    }
}

save_json_metrics(export_data, output_path)
print(f"\nExperiment 09 results saved to: {output_path}")

Tier                           | Ranks           | Params     | Ratio  | Cut %   | Err %    | Accuracy  | Delta  
Baseline (Uncompressed)        | Full            | 207028224  | 1.00x  | 0.00%   | 0.00%    | 48.00%    | +0.00% 
Tier 1 (Ultra ~99% Cut)        | [2, 20, 20]     | 135971576  | 1.52x  | 34.32%  | 97.84%   | 34.10%    | -13.90%
Tier 2 (Aggressive ~89% Cut)   | [3, 80, 200]    | 143214292  | 1.45x  | 30.82%  | 90.55%   | 37.10%    | -10.90%
Tier 2.5 (Target ~77% Cut)     | [4, 120, 360]   | 151667568  | 1.37x  | 26.74%  | 83.55%   | 32.00%    | -16.00%
Tier 3 (Moderate ~57% Cut)     | [4, 180, 600]   | 166219248  | 1.25x  | 19.71%  | 76.30%   | 32.90%    | -15.10%
cummulative_time: 334.71s

Experiment 09 results saved to: artifacts/09_all_layers_gd_branch1_results.json
time: 0.01s
cummulative_time: 334.71s
